In [43]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# config          # configures the model run with input data and defaults (where not directly input)
# spatial_cv      # set up the spatial CV blocks based on fold CSV
# train           # trains model and predicts on test data
# evaluate        # calculates metrics for each fold and overall 
# importance      # calculates feature importance + permutation importance for each fold and overall 

from pathlib import Path
import json
import dataclasses
import pandas as pd

from step_a_config import RunConfig
from step_b_spatial_cv import make_spatial_folds
from step_c_train import train_model
from step_d_evaluate import evaluate
from step_e_importance import get_feature_importance

In [44]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [45]:
#### DEFINE RUN (MANUAL)

version = 'capital' # capital or labor
unit = 'tonne' # USD or tonne
type_of_model = 'rf' # rf or qrf
name_of_run =  f'{version}_{unit}_{type_of_model}_v2' 

In [46]:
##### Configuration (doesn't not change)

# set target variable 
unit_map = {
    ("capital", "tonne"): "tonne",
    ("capital", "USD"): "USD",
    ("labor", "tonne"): "tonne",
    ("labor", "USD"): "million_USD",
}
unit_final = unit_map[(version, unit)]

numerator = "USD" if version == "capital" else "jobs"
country_labor_unit = "million_USD" if unit == "USD" else "tonne"

target_var = f'log_{version}_intensity_{numerator}_per_{unit_final}' 
country_capital_var = f'log_country_capital_intensity_USD_per_{unit}' 
country_labor_var = f'log_country_labor_intensity_jobs_per_{country_labor_unit}' 

# config
fold = FOLD_DIR / f"{version}_folds.csv"
model_data = f"{version}_final.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID"],
    ),
]

# define which columns are feature columns 
feature_cols = ['SOC', 'pct_cropland_irrigated', 'female_share',
       'pop_density_people_per_km2', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'slope',
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'pct_GDP_ag', 'share_large_field', 'share_medium_field',
       'share_small_field', 'share_vsmall_field', 'lon', 'lat',
       'share_with_nightlights', 'crop_intensity', 'log_tonnes_production_per_HA',
       'log_USD_production_per_HA', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2',
       country_capital_var, country_labor_var]


In [47]:
##### Define function to save results into defined directory
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True) # creates folder if it doesn't already exist 

    # save config details for run 
    # convert any Path objects to strings so json can handle them
    config_dict = dataclasses.asdict(config)
    config_dict = {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config_dict.items()
    }
    (out_dir / "config.json").write_text(
        json.dumps(config_dict, indent=2)
    )

    # save predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # save the best parameters for each fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # save metrics and importance scores
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")

In [ ]:
##### RUN   

# define the function to run all scripts in order 
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load model data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # run spatial folds functions 
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # run training and prediction functions
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # run evaluation functions
    results["metrics"] = evaluate(results, config)

    # run importance functions
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # run save function
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)

# Run run function for each model configuration defined above
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: capital_tonne_rf_v2
  target: log_capital_intensity_USD_per_tonne  |  model: rf
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
  fold_1: 25 train countries (4,929 rows) | 1 test countries (5,182 rows)
  fold_2: 25 train countries (8,827 rows) | 1 test countries (1,284 rows)
  fold_3: 25 train countries (7,403 rows) | 1 test countries (2,708 rows)
  fold_4: 23 train countries (937 rows) | 3 test countries (9,174 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 500, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 10}

── fold_2 ──────────────────────────────
  test countries: ['CAN']
  best params: {'n_estimators': 200, 'min_samples_leaf': 5, 'max_features': 0.3, 'max_depth': None}

── fold_3 ──────────────────────────────